# 🧪 Fine-Tuning Eficiente de um Modelo de Linguagem com LoRA (PEFT)

Este tutorial conduz você, passo a passo, pelo processo de **ajuste fino (fine-tuning)** de um modelo de linguagem causal (ex.: `distilgpt2`) utilizando **LoRA** (*Low-Rank Adaptation*), uma das técnicas de **PEFT** (*Parameter-Efficient Fine-Tuning*).  
Ao final, você compreenderá os fundamentos matemáticos por trás do método e saberá como adaptar um modelo grande com muito menos recursos computacionais.

**O que você vai aprender:**
- A diferença entre *full fine-tuning* e *PEFT*.
- A ideia central do LoRA (matrizes de baixo posto).
- Como preparar dados, configurar, treinar e testar um modelo com LoRA.
- Comparar a qualidade das respostas antes e depois do ajuste fino.

## 📚 1. Por que Fine-Tuning Eficiente?

Modelos de linguagem modernos possuem bilhões de parâmetros. Atualizar **todos** os pesos durante o treinamento (*full fine-tuning*) exige:
- GPUs com dezenas de GB de memória.
- Armazenamento de uma cópia completa do modelo para cada tarefa.

**PEFT (Parameter-Efficient Fine-Tuning)** resolve esse problema treinando apenas um pequeno conjunto de **novos parâmetros**, mantendo o modelo base congelado.  

### 🔹 LoRA (Low-Rank Adaptation)
A hipótese do LoRA é que as atualizações dos pesos durante o fine-tuning possuem uma **estrutura de baixo posto** (*low intrinsic rank*).  
Assim, em vez de aprender a matriz completa de atualização $\Delta W \in \mathbb{R}^{d \times k}$, aprendemos duas matrizes menores:

$$\Delta W = B \cdot A$$

onde:
- $B \in \mathbb{R}^{d \times r}$
- $A \in \mathbb{R}^{r \times k}$
- $r \ll \min(d, k)$ (o **rank** da adaptação)

O número de parâmetros treináveis cai de $d \times k$ para $r \times (d + k)$, uma redução drástica quando $r$ é pequeno.

### 🔹 Como isso é usado na prática?
Durante o treinamento, a saída de uma camada linear original $h = W x$ é modificada para:

$$h = W x + \Delta W x = W x + B A x$$

A matriz $A$ é inicializada com uma distribuição gaussiana e $B$ com zeros, de forma que no início $\Delta W = 0$.  
Um fator de escala $\alpha$ controla a intensidade da adaptação; frequentemente a atualização é escalada por $\frac{\alpha}{r}$:

$$h = W x + \frac{\alpha}{r} B A x$$

Após o treinamento, podemos **fundir** (*merge*) os pesos adaptados ao modelo original: $W_{\text{merged}} = W + \frac{\alpha}{r} BA$, eliminando qualquer custo extra na inferência.

Neste notebook, usaremos a biblioteca `peft` (Hugging Face) para aplicar LoRA ao `distilgpt2`.

## 📦 2. Requisitos

Execute o comando abaixo para instalar as dependências necessárias (descomente a linha caso ainda não estejam instaladas):

In [3]:
!pip install transformers datasets peft accelerate torch bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00


In [2]:
%cd /content/llm-finetuning-rag-system

import sys
sys.path.append("/content/llm-finetuning-rag-system")

/content/llm-finetuning-rag-system


Importe os módulos que serão utilizados ao longo do processo:

In [4]:
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
import torch

## 🤖 3. Carregar o Modelo Pré-Treinado e o Tokenizador

Vamos carregar o `distilgpt2` – uma versão menor e mais rápida do GPT-2, ideal para experimentação.  
Como o tokenizador original não define um `pad_token`, usaremos o `eos_token` no lugar.

In [5]:
model_name = "microsoft/Phi-4-mini-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Modelo carregado: {model_name}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/2.93k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/16.3k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Modelo carregado: microsoft/Phi-4-mini-instruct


## 🔢 4. Carregar e Preparar o Dataset

Utilizaremos um arquivo `dataset.jsonl` onde cada linha contém uma instrução (`instruction`) e a saída desejada (`output`).  
Vamos converter cada exemplo em uma única string no formato:
```
Instruction: <instrução>
Output: <saída>
```
e depois dividir o conjunto em treino (80%) e validação (20%).

In [6]:
def convert_to_hf_format(example):
    """Aplica o Chat Template oficial do Phi-4 para unir instrução e saída."""
    messages = [
        {"role": "user", "content": example["Instruction"]},
        {"role": "assistant", "content": example["Output"]}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

# Carrega o arquivo JSON Lines
dataset = load_dataset('json', data_files='data/processed/dataset_curado_lora_agressivo.jsonl')
# Aplica a conversão
dataset = dataset.map(convert_to_hf_format)
# Divide em treino e teste
dataset = dataset["train"].train_test_split(test_size=0.2)
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/138 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Instruction', 'Output', 'text'],
        num_rows: 110
    })
    test: Dataset({
        features: ['Instruction', 'Output', 'text'],
        num_rows: 28
    })
})


## 🧪 5. Inferência ANTES do Fine-Tuning

Antes de qualquer treinamento, vamos ver como o modelo base responde a uma pergunta que está no nosso dataset.  
Isso servirá como **linha de base** para compararmos com o modelo ajustado.

In [17]:
def generate_response(model, tokenizer, instruction, input_text=""):
    """Gera uma resposta a partir de uma instrução, usando o modelo fornecido."""
    messages = [{"role": "user", "content": instruction}]
    if input_text:
        messages = [{"role": "user", "content": f"{instruction}\nContexto: {input_text}"}]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=128,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,          # ativa amostragem para usar temperatura
        temperature=0.3
    )
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    resposta = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    return resposta

# Exemplo de instrução (deve existir no dataset.jsonl)
test_instruction = "Qual é o impacto da privação de sono na função cerebral?"

print("=== ANTES DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta base: {generate_response(base_model, tokenizer, test_instruction)}")

=== ANTES DO FINE-TUNING ===
Instrução: Qual é o impacto da privação de sono na função cerebral?


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Resposta base: A privação de sono afeta a função cerebral, alterando a atividade cerebral, o desempenho cognitivo e a função de alerta.


> **Observação:** O modelo base provavelmente gerará um texto genérico ou sem relação direta com a instrução, pois ainda não foi adaptado ao nosso domínio.

## ✂️ 6. Tokenização do Dataset

Transformamos os textos em sequências de tokens que o modelo pode processar.  
Usaremos `padding="max_length"` e `truncation=True` para garantir que todas as amostras tenham o mesmo comprimento (128 tokens).

In [8]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding=False
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)
print("Dataset tokenizado:", tokenized_datasets)

Map:   0%|          | 0/110 [00:00<?, ? examples/s]

Map:   0%|          | 0/28 [00:00<?, ? examples/s]

Dataset tokenizado: DatasetDict({
    train: Dataset({
        features: ['Instruction', 'Output', 'text', 'input_ids', 'attention_mask'],
        num_rows: 110
    })
    test: Dataset({
        features: ['Instruction', 'Output', 'text', 'input_ids', 'attention_mask'],
        num_rows: 28
    })
})


## 🔧 7. Preparar o Modelo para LoRA

A função `prepare_model_for_kbit_training` ativa técnicas como *gradient checkpointing* e ajusta a arquitetura para treinamento eficiente.  
É essencial quando se utiliza quantização (QLoRA), mas também é recomendada mesmo sem quantização para melhor gerenciamento de memória.

In [9]:
# A partir daqui, usaremos uma cópia do modelo base para o fine-tuning
model = base_model  # poderia também ser uma nova instância
model = prepare_model_for_kbit_training(model)

## 🧩 8. Configurar e Injetar LoRA

Agora definimos a configuração do LoRA:
- **r**: posto das matrizes de adaptação (quanto maior, mais capacidade, porém mais parâmetros).
- **lora_alpha**: fator de escala $\alpha$ (a atualização será multiplicada por $\alpha / r$).
- **target_modules**: os módulos do transformer onde aplicaremos LoRA. No GPT-2, `c_attn` e `c_proj` são as projeções de atenção.
- **lora_dropout**: dropout aplicado às matrizes LoRA para regularização.
- **bias**: "none" significa que não treinamos os vieses.
- **task_type**: como é um modelo de linguagem causal, usamos `CAUSAL_LM`.

Em seguida, criamos o modelo PEFT com `get_peft_model`, que insere os adaptadores LoRA e **congela** o resto do modelo.

In [10]:
lora_config = LoraConfig(
    r=16,                    # rank da decomposição
    lora_alpha=32,           # escala alpha
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],  # camadas ,alvo
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    inference_mode=False     # False = modo treinamento
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,145,728 || all params: 3,839,167,488 || trainable%: 0.0819


✅ **Interpretação:** Apenas uma fração mínima do total de parâmetros será atualizada.  
No exemplo, menos de 1% dos pesos são treináveis – é a essência do PEFT.

## 🧱 9. Data Collator para Modelagem Causal

O `DataCollatorForLanguageModeling` prepara os lotes para o treinamento de linguagem causal (sem *masked language modeling*).  
Ele automaticamente desloca os rótulos para que a tarefa seja prever o próximo token.

In [11]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

## ⚙️ 10. Argumentos de Treinamento

Definimos os hiperparâmetros do treinamento.  
Como nosso dataset é pequeno, usaremos 100 épocas e uma taxa de aprendizado relativamente alta (`1e-3`).  
O `eval_steps` controla a frequência da avaliação no conjunto de validação.

In [13]:
training_args = TrainingArguments(
    output_dir="lora_models/causal_model_1",
    num_train_epochs=3,
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    weight_decay=0.01,
    logging_steps=10,
    fp16=True,
    save_total_limit=2,
    report_to="none"
)

## 🏋️ 11. Inicializar o Trainer

O `Trainer` do Hugging Face orquestra todo o ciclo de treinamento, avaliação e salvamento.

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
)

## 🚀 12. Treinar o Modelo

Iniciamos o treinamento. Acompanhe a perda (*loss*) nos logs – ela deve diminuir ao longo das épocas.

In [15]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,1.784668,1.900995
2,1.632286,1.872698
3,1.778294,1.903182


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=165, training_loss=1.9554979382139264, metrics={'train_runtime': 122.3564, 'train_samples_per_second': 2.697, 'train_steps_per_second': 1.349, 'total_flos': 339082975457280.0, 'train_loss': 1.9554979382139264, 'epoch': 3.0})

## 💾 13. Salvar o Modelo Ajustado e o Tokenizador

Ao final do treinamento, salvamos os pesos LoRA (apenas os adaptadores) e o tokenizador.

In [16]:
model.save_pretrained("lora_models/causal_model_1/final_adapter")
tokenizer.save_pretrained("lora_models/causal_model_1/final_tokenizer")

('lora_models/causal_model_1/final_tokenizer/tokenizer_config.json',
 'lora_models/causal_model_1/final_tokenizer/chat_template.jinja',
 'lora_models/causal_model_1/final_tokenizer/tokenizer.json')

## 💻 14. Inferência APÓS o Fine-Tuning

Agora carregamos o modelo ajustado e comparamos sua resposta com a versão base, usando **exatamente a mesma instrução**.

In [18]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

finetuned_model = PeftModel.from_pretrained(
    base_model,
    "lora_models/causal_model_1/final_adapter"
)

finetuned_tokenizer = AutoTokenizer.from_pretrained(
    "lora_models/causal_model_1/final_tokenizer"
)

if finetuned_tokenizer.pad_token is None:
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

In [19]:
print("\n=== DEPOIS DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")

# Executa a função usando as tags de chat do Phi-4
resposta_ajustada = generate_response(finetuned_model, finetuned_tokenizer, test_instruction)
print(f"Resposta ajustada: {resposta_ajustada}")


=== DEPOIS DO FINE-TUNING ===
Instrução: Qual é o impacto da privação de sono na função cerebral?
Resposta ajustada: A privação de sono afeta negativamente a função cerebral.


## 📊 15. Comparação e Conclusão

- **Antes do fine-tuning:** o modelo base não conhecia nosso domínio; sua resposta era genérica ou incoerente.
- **Depois do fine-tuning:** com apenas uma fração dos parâmetros treinados (via LoRA), o modelo aprendeu a estrutura desejada e gera respostas alinhadas com os exemplos fornecidos.

Esse é o poder do **PEFT**: adaptar grandes modelos de forma rápida, barata e com resultados surpreendentes.

### 📌 Resumo dos conceitos-chave

| Conceito | Descrição |
|----------|-----------|
| **Full fine-tuning** | Atualiza todos os pesos do modelo. |
| **PEFT** | Atualiza apenas um pequeno número de parâmetros novos. |
| **LoRA** | Decompõe a atualização $\Delta W$ em $B A$, com $r \ll \min(d,k)$. |
| **r** | Posto da decomposição – controla a capacidade da adaptação. |
| **$\alpha$** | Fator de escala que ajusta a intensidade da adaptação. |
| **Target modules** | Camadas onde os adaptadores LoRA são inseridos. |

Agora você pode experimentar com outros valores de `r`, `lora_alpha`, ou até mesmo outros modelos!